In [ ]:
import warnings
import os
import requests
import tarfile
import zipfile
from tqdm import tqdm
import torch
import cv2
import torch_directml
import torchvision
import pandas as pd
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor
from sklearn.metrics import accuracy_score
from torchvision.datasets import VisionDataset
from torchvision.io import read_image
from torchvision.utils import make_grid
from torch.optim.lr_scheduler import StepLR
from PIL import Image
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [ ]:
import sys
print(sys.executable)

import torch_directml
device = torch_directml.device()
print("DirectML device:", device)


In [ ]:
# Suppress DeprecationWarnings from tarfile
warnings.filterwarnings("ignore", category=DeprecationWarning)



# Set working directory
working_dir = 'C:/Kvasir'
os.makedirs(working_dir, exist_ok=True)
os.chdir(working_dir)

# Updated dataset URL
dataset_url = "https://datasets.simula.no/downloads/kvasir-instrument.zip"
dataset_zip = os.path.join(working_dir, "kvasir-instrument.zip")
dataset_dir = os.path.join(working_dir, "kvasir-instrument")

In [ ]:
# Function to download the dataset with error checking
def download_dataset(url, save_path):
    if not os.path.exists(save_path):
        print("Downloading dataset...")
        response = requests.get(url, stream=True)
        total_size = int(response.headers.get('content-length', 0))
        if response.status_code == 200:
            with open(save_path, "wb") as f, tqdm(
                desc="Downloading " + os.path.basename(save_path),
                total=total_size,
                unit='iB',
                unit_scale=True,
                unit_divisor=1024,
            ) as bar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))
        else:
            raise Exception(f"Failed to download file: status code {response.status_code}")

In [ ]:
# Download the dataset
try:
    download_dataset(dataset_url, dataset_zip)
except Exception as e:
    print(str(e))
    print("Please check the dataset URL or your internet connection.")
    exit(1)

# Extract the dataset
if not os.path.exists(dataset_dir):
    try:
        with zipfile.ZipFile(dataset_zip, "r") as zip_ref:
            print("Extracting dataset zip file...")
            zip_ref.extractall(working_dir)
    except zipfile.BadZipFile:
        print("Error: The downloaded file is not a valid zip file.")
        print("Deleting corrupted file and retrying...")
        os.remove(dataset_zip)
        raise

print("Dataset ready!")

In [ ]:
# Extract images and masks tar files
images_tar = os.path.join(dataset_dir, 'images.tar.gz')
masks_tar = os.path.join(dataset_dir, 'masks.tar.gz')

if os.path.exists(images_tar):
    print("Extracting images tar file...")
    with tarfile.open(images_tar, "r:gz") as tar:
        tar.extractall(path=os.path.join(dataset_dir, 'images'))

if os.path.exists(masks_tar):
    print("Extracting masks tar file...")
    with tarfile.open(masks_tar, "r:gz") as tar:
        tar.extractall(path=os.path.join(dataset_dir, 'masks'))

print("Extraction complete.")

In [ ]:
# Dataset directories
images_dir = os.path.join(dataset_dir, 'images', 'images')
masks_dir = os.path.join(dataset_dir, 'masks', 'masks')

# Option to use provided splits or custom split
use_provided_splits = True  # Set to False to use custom split

if use_provided_splits:
    # Paths to the dataset files
    train_txt_path = os.path.join(dataset_dir, 'train.txt')
    test_txt_path = os.path.join(dataset_dir, 'test.txt')

    # Read train and test splits
    with open(train_txt_path, 'r') as f:
        train_files = f.read().splitlines()
    with open(test_txt_path, 'r') as f:
        test_files = f.read().splitlines()

    # Construct file paths
    train_images_list = [os.path.join(images_dir, f"{img_id}.jpg") for img_id in train_files]
    train_masks_list = [os.path.join(masks_dir, f"{img_id}.png") for img_id in train_files]
    test_images_list = [os.path.join(images_dir, f"{img_id}.jpg") for img_id in test_files]
    test_masks_list = [os.path.join(masks_dir, f"{img_id}.png") for img_id in test_files]

else:
    # Custom train/test split
    # Adjust train_ratio to change the size of the training data
    train_ratio = 0.7  # 70% for training, adjust as needed

    # Get all image and mask filenames
    all_images = sorted(os.listdir(images_dir))
    all_masks = sorted(os.listdir(masks_dir))

    # Ensure the number of images and masks are equal
    assert len(all_images) == len(all_masks), "Mismatch in number of images and masks"

    # Create the custom split
    total_samples = len(all_images)
    train_size = int(train_ratio * total_samples)
    train_images_list = [os.path.join(images_dir, img) for img in all_images[:train_size]]
    train_masks_list = [os.path.join(masks_dir, mask) for mask in all_masks[:train_size]]
    test_images_list = [os.path.join(images_dir, img) for img in all_images[train_size:]]
    test_masks_list = [os.path.join(masks_dir, mask) for mask in all_masks[train_size:]]

In [ ]:
# Verify that images and masks are correctly paired
print("Sample image and mask paths:")
for img_path, mask_path in zip(train_images_list[:3], train_masks_list[:3]):
    print(f"Image: {img_path}, Mask: {mask_path}")

# Dataset class
class KvasirInstrumentDataset(Dataset):
    def __init__(self, images_list, masks_list, augmentations=None):
        self.images_list = images_list
        self.masks_list = masks_list
        self.augmentations = augmentations

    def __len__(self):
        return len(self.images_list)

    def __getitem__(self, idx):
        image_path = self.images_list[idx]
        mask_path = self.masks_list[idx]

        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Image not found: {image_path}")

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise ValueError(f"Mask not found: {mask_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = (mask > 0).astype(np.float32)

        if self.augmentations:
            augmented = self.augmentations(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        else:
            transform = A.Compose([A.Resize(256, 256), A.Normalize(), ToTensorV2()])
            transformed = transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']

        # Ensure mask has shape [1, H, W]
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        elif mask.ndim == 3 and mask.shape[0] != 1:
            mask = mask.unsqueeze(0)

        return image, mask

In [ ]:

# Define UNet model (same as before)
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, init_features=32):
        super(UNet, self).__init__()

        features = init_features
        self.encoder1 = self._block(in_channels, features)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder2 = self._block(features, features * 2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder3 = self._block(features * 2, features * 4)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder4 = self._block(features * 4, features * 8)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = self._block(features * 8, features * 16)

        self.upconv4 = nn.ConvTranspose2d(features * 16, features * 8, kernel_size=2, stride=2)
        self.decoder4 = self._block((features * 8) * 2, features * 8)
        self.upconv3 = nn.ConvTranspose2d(features * 8, features * 4, kernel_size=2, stride=2)
        self.decoder3 = self._block((features * 4) * 2, features * 4)
        self.upconv2 = nn.ConvTranspose2d(features * 4, features * 2, kernel_size=2, stride=2)
        self.decoder2 = self._block((features * 2) * 2, features * 2)
        self.upconv1 = nn.ConvTranspose2d(features * 2, features, kernel_size=2, stride=2)
        self.decoder1 = self._block(features * 2, features)

        self.conv = nn.Conv2d(features, out_channels, kernel_size=1)

    def _block(self, in_channels, features):
        return nn.Sequential(
            nn.Conv2d(in_channels, features, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(features),
            nn.ReLU(inplace=True),
            nn.Conv2d(features, features, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(features),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool1(enc1))
        enc3 = self.encoder3(self.pool2(enc2))
        enc4 = self.encoder4(self.pool3(enc3))

        bottleneck = self.bottleneck(self.pool4(enc4))

        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.decoder4(dec4)
        dec3 = self.upconv3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.decoder3(dec3)
        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.decoder2(dec2)
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.decoder1(dec1)
        return torch.sigmoid(self.conv(dec1))

In [ ]:


# UNet++ Decoder with Nested Skip Connections
class UNetPlusPlusDecoder(nn.Module):
    def __init__(self, encoder_channels, decoder_channels, activation=nn.ReLU()):
        super(UNetPlusPlusDecoder, self).__init__()
        self.activation = activation

        # The encoder_channels are expected in the order: [layer4, layer3, layer2, layer1]
        self.conv0_0 = self.conv_block(encoder_channels[0], decoder_channels[0])
        self.conv1_0 = self.conv_block(encoder_channels[1], decoder_channels[1])
        self.conv1_1 = self.conv_block(decoder_channels[0] + decoder_channels[1], decoder_channels[1])
        self.conv2_0 = self.conv_block(encoder_channels[2], decoder_channels[2])
        self.conv2_1 = self.conv_block(decoder_channels[1] + decoder_channels[2], decoder_channels[2])
        self.conv2_2 = self.conv_block(decoder_channels[0] + decoder_channels[1] + decoder_channels[2], decoder_channels[2])

    def conv_block(self, in_channels, out_channels):
        """Creates a block of Conv2d -> BatchNorm2d -> Activation."""
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            self.activation
        )

    def upsample_to_size(self, x, target_size):
        return F.interpolate(x, size=target_size, mode='bilinear', align_corners=True)

    def forward(self, x4, x3, x2, x1):
        # x4: deepest encoder feature (e.g., ResNet layer4: 512 channels)
        # x3: ResNet layer3 (256 channels)
        # x2: ResNet layer2 (128 channels)
        # x1: ResNet layer1 (64 channels)
        x0_0 = self.conv0_0(x4)  # Process deepest feature
        x1_0 = self.conv1_0(self.upsample_to_size(x3, x2.shape[2:]))  # Upsample x3 to x2's size
        x1_1 = self.conv1_1(torch.cat([x1_0, self.upsample_to_size(x0_0, x1_0.shape[2:])], dim=1))
        x2_0 = self.conv2_0(self.upsample_to_size(x2, x1_0.shape[2:]))  # Upsample x2 to x1_0's size
        x2_1 = self.conv2_1(torch.cat([x2_0, self.upsample_to_size(x1_1, x2_0.shape[2:])], dim=1))
        x2_2 = self.conv2_2(torch.cat([
            x2_1, 
            self.upsample_to_size(x1_1, x2_1.shape[2:]), 
            self.upsample_to_size(x0_0, x2_1.shape[2:])
        ], dim=1))
        return x2_2

# UNet++ Model for Binary Segmentation
class UNetPlusPlus(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, activation=nn.ReLU()):
        super(UNetPlusPlus, self).__init__()
        # Use a pretrained ResNet34 as the encoder
        self.encoder = torchvision.models.resnet34(pretrained=True)
        # Optionally, freeze the encoder parameters:
        # for param in self.encoder.parameters():
        #     param.requires_grad = False

        # ResNet34 produces:
        # - layer1: 64 channels
        # - layer2: 128 channels
        # - layer3: 256 channels
        # - layer4: 512 channels
        # The decoder expects encoder features in the order: [layer4, layer3, layer2, layer1]
        encoder_channels = [512, 256, 128, 64]
        # Define decoder channels (only first three are used in this decoder design)
        decoder_channels = [256, 128, 64, 32]
        self.decoder = UNetPlusPlusDecoder(encoder_channels, decoder_channels, activation)
        self.segmentation_head = nn.Conv2d(decoder_channels[2], out_channels, kernel_size=3, padding=1)

    def forward(self, x):
        # Save original input size for output interpolation
        input_size = x.shape[2:]
        
        # If needed, normalize input for ResNet (using ImageNet statistics)
        # mean = torch.tensor([0.485, 0.456, 0.406]).to(x.device).view(1,3,1,1)
        # std = torch.tensor([0.229, 0.224, 0.225]).to(x.device).view(1,3,1,1)
        # x = (x - mean) / std

        # Encoder forward pass:
        x = self.encoder.conv1(x)
        x = self.encoder.bn1(x)
        x = self.encoder.relu(x)
        x = self.encoder.maxpool(x)

        x1 = self.encoder.layer1(x)  # 64 channels
        x2 = self.encoder.layer2(x1)  # 128 channels
        x3 = self.encoder.layer3(x2)  # 256 channels
        x4 = self.encoder.layer4(x3)  # 512 channels

        # Pass features to the decoder in the order: x4, x3, x2, x1
        x = self.decoder(x4, x3, x2, x1)
        x = self.segmentation_head(x)
        # Interpolate to match the original input size dynamically
        x = F.interpolate(x, size=input_size, mode='bilinear', align_corners=True)
        return torch.sigmoid(x)


In [ ]:

# Convolution block with 3x3 kernel size, Group Normalization, and ReLU activation
class Conv3x3GNReLU(nn.Module):
    def __init__(self, in_channels, out_channels, upsample=False):
        super().__init__()
        self.upsample = upsample
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, (3, 3), stride=1, padding=1, bias=False),
            nn.GroupNorm(32, out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.block(x)
        if self.upsample:
            x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=True)
        return x

# Feature Pyramid Network block
class FPNBlock(nn.Module):
    def __init__(self, pyramid_channels, skip_channels):
        super().__init__()
        self.skip_conv = nn.Conv2d(skip_channels, pyramid_channels, kernel_size=1, bias=False)

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        if skip is not None:
            skip = self.skip_conv(skip)
            skip = F.interpolate(skip, size=x.shape[2:], mode='bilinear', align_corners=True)
            x = x + skip
        return x

# Segmentation block with a series of convolutions and upsampling
class SegmentationBlock(nn.Module):
    def __init__(self, in_channels, out_channels, n_upsamples=0):
        super().__init__()
        blocks = [Conv3x3GNReLU(in_channels, out_channels, upsample=bool(n_upsamples))]

        if n_upsamples > 1:
            for _ in range(1, n_upsamples):
                blocks.append(Conv3x3GNReLU(out_channels, out_channels, upsample=True))

        self.block = nn.Sequential(*blocks)

    def forward(self, x):
        return self.block(x)

# Merge block to combine features using addition or concatenation
class MergeBlock(nn.Module):
    def __init__(self, policy='add'):
        super().__init__()
        self.policy = policy

    def forward(self, x):
        if self.policy == 'add':
            return sum(x)
        elif self.policy == 'cat':
            return torch.cat(x, dim=1)
        else:
            raise ValueError("`merge_policy` must be one of: ['add', 'cat']")

# FPN Decoder combining the feature pyramid with skip connections
class FPNDecoder(nn.Module):
    def __init__(self, encoder_channels, pyramid_channels=256, segmentation_channels=128, merge_policy='add'):
        super().__init__()

        self.p5 = nn.Conv2d(encoder_channels[0], pyramid_channels, kernel_size=1)
        self.p4 = FPNBlock(pyramid_channels, encoder_channels[1])
        self.p3 = FPNBlock(pyramid_channels, encoder_channels[2])
        self.p2 = FPNBlock(pyramid_channels, encoder_channels[3])

        self.seg_blocks = nn.ModuleList([
            SegmentationBlock(pyramid_channels, segmentation_channels, n_upsamples=n_upsamples)
            for n_upsamples in [3, 2, 1, 0]
        ])

        self.merge = MergeBlock(merge_policy)

    def forward(self, *features):
        # Reverse the features to match the FPN order
        features = features[::-1]  # Now features[0] is the deepest (layer4)
        p5 = self.p5(features[0])
        p4 = self.p4(p5, features[1])
        p3 = self.p3(p4, features[2])
        p2 = self.p2(p3, features[3])

        feature_pyramid = [seg_block(p) for seg_block, p in zip(self.seg_blocks, [p5, p4, p3, p2])]
        x = self.merge(feature_pyramid)

        return x

# FPN main class
class FPN(nn.Module):
    def __init__(self, encoder_name='resnet34', in_channels=3, decoder_pyramid_channels=256, segmentation_channels=128, out_channels=1, activation_fn=nn.ReLU(), pretrained=True):
        super(FPN, self).__init__()

        if encoder_name not in ['resnet34', 'resnet50']:
            raise ValueError(f"Encoder {encoder_name} is not supported")

        if encoder_name == 'resnet34':
            self.encoder = torchvision.models.resnet34(pretrained=pretrained)
            # For ResNet34, expected encoder feature channels (from later layers):
            # layer4: 512, layer3: 256, layer2: 128, layer1: 64
            encoder_channels = [512, 256, 128, 64]
        elif encoder_name == 'resnet50':
            self.encoder = torchvision.models.resnet50(pretrained=pretrained)
            # For ResNet50:
            encoder_channels = [2048, 1024, 512, 256]

        if in_channels != 3:
            self.encoder.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)

        self.decoder = FPNDecoder(encoder_channels, pyramid_channels=decoder_pyramid_channels, segmentation_channels=segmentation_channels, merge_policy='add')
        self.segmentation_head = nn.Conv2d(segmentation_channels, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        features = []
        # Run the encoder and collect features
        # Initial layers: conv1 -> bn1 -> relu -> maxpool
        x = self.encoder.conv1(x)
        x = self.encoder.bn1(x)
        x = self.encoder.relu(x)
        x = self.encoder.maxpool(x)
        # Now collect features from layers 1 to 4
        x1 = self.encoder.layer1(x)  # 64 channels
        x2 = self.encoder.layer2(x1)  # 128 channels
        x3 = self.encoder.layer3(x2)  # 256 channels
        x4 = self.encoder.layer4(x3)  # 512 channels (or 2048 for ResNet50)

        # Assemble features in the order [layer4, layer3, layer2, layer1]
        features = [x1, x2, x3, x4]

        # Pass the features through the FPN decoder
        x = self.decoder(*features)
        # Pass the result through the segmentation head
        x = self.segmentation_head(x)
        # Dynamically upscale to the original input size
        x = F.interpolate(x, size=input_size, mode='bilinear', align_corners=True)
        return torch.sigmoid(x)


In [ ]:
# Define augmentations
augmentations = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.GaussNoise(p=0.2),
    A.Resize(256, 256),
    A.Normalize(),
    ToTensorV2()
])

# No augmentations (only resize, normalize, and convert to tensor)
no_augmentations = A.Compose([
    A.Resize(256, 256),
    A.Normalize(),
    ToTensorV2()
])

# Training process functions (same as before)
def compute_metrics(preds, targets, smooth=1e-6):
    preds = preds.view(-1).cpu().numpy()
    targets = targets.view(-1).cpu().numpy()
    intersection = (preds * targets).sum()
    union = preds.sum() + targets.sum() - intersection
    dice = (2. * intersection + smooth) / (preds.sum() + targets.sum() + smooth)
    iou = (intersection + smooth) / (union + smooth)
    acc = accuracy_score(targets, preds)
    return dice, iou, acc

In [ ]:
def train_model(model, 
                train_loader, 
                val_loader,  
                optimizer, 
                scheduler, 
                criterion, 
                num_epochs=20, 
                patience=5):
    history = {
        'train_loss': [], 'val_loss': [],
        'train_dice': [], 'val_dice': [],
        'train_iou': [], 'val_iou': [],
        'train_acc': [], 'val_acc': [],
        'test_loss': [], 'test_dice': [],
        'test_iou': [], 'test_acc': []
    }
    
    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(num_epochs):
        # ----------------- Training Phase ----------------- #
        model.train()
        train_loss, train_dice, train_iou, train_acc = 0, 0, 0, 0

        for images, masks in tqdm(train_loader, desc=f"Training Epoch {epoch+1}/{num_epochs}", unit="batch", leave=False):
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            preds = (outputs > 0.5).float()
            dice, iou, acc = compute_metrics(preds, masks)
            train_loss += loss.item()
            train_dice += dice
            train_iou += iou
            train_acc += acc

        train_loss /= len(train_loader)
        train_dice /= len(train_loader)
        train_iou  /= len(train_loader)
        train_acc  /= len(train_loader)

        # ----------------- Validation Phase ----------------- #
        val_loss, val_dice, val_iou, val_acc = 0, 0, 0, 0
        model.eval()
        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f"Validation Epoch {epoch+1}/{num_epochs}", unit="batch", leave=False):
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)
                loss = criterion(outputs, masks)
                preds = (outputs > 0.5).float()
                dice, iou, acc = compute_metrics(preds, masks)
                val_loss += loss.item()
                val_dice += dice
                val_iou += iou
                val_acc += acc

        val_loss /= len(val_loader)
        val_dice /= len(val_loader)
        val_iou  /= len(val_loader)
        val_acc  /= len(val_loader)

        # # ----------------- Test Phase ----------------- #
        # test_loss, test_dice, test_iou, test_acc = 0, 0, 0, 0
        # model.eval()
        # with torch.no_grad():
        #     for images, masks in tqdm(test_loader, desc=f"Test Epoch {epoch+1}/{num_epochs}", unit="batch", leave=False):
        #         images, masks = images.to(device), masks.to(device)
        #         outputs = model(images)
        #         loss = criterion(outputs, masks)
        #         preds = (outputs > 0.5).float()
        #         dice, iou, acc = compute_metrics(preds, masks)
        #         test_loss += loss.item()
        #         test_dice += dice
        #         test_iou += iou
        #         test_acc += acc
        # 
        # test_loss /= len(test_loader)
        # test_dice /= len(test_loader)
        # test_iou  /= len(test_loader)
        # test_acc  /= len(test_loader)

        # ----------------- Logging / History ----------------- #
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_dice'].append(train_dice)
        history['val_dice'].append(val_dice)
        history['train_iou'].append(train_iou)
        history['val_iou'].append(val_iou)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        # ----------------- Scheduler & Early Stopping ----------------- #
        scheduler.step(val_loss)
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Current Learning Rate: {current_lr}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(working_dir, 'best_model.pth'))
        else:
            patience_counter += 1

        print(f"EarlyStopping counter: {patience_counter} out of {patience}")
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

        # ----------------- Print Epoch Metrics ----------------- #
        print(f"Epoch {epoch + 1}/{num_epochs} - "
              f"Training Loss: {train_loss:.4f} - "
              f"Validation Loss: {val_loss:.4f} - "
              f"Training Acc: {train_acc*100:.2f}% - "
              f"Validation Acc: {val_acc*100:.2f}% - "
              f"Training Dice: {train_dice:.4f} - "
              f"Validation Dice: {val_dice:.4f} - "
              f"Training IoU: {train_iou:.4f} - "
              f"Validation IoU: {val_iou:.4f} ")

        # Visualize predictions every 10 epochs
        if (epoch + 1) % 10 == 0:
            visualize_predictions(model, val_loader)

    return history


In [ ]:
def visualize_predictions(model, dataloader, num_samples=3):
    model.eval()
    images, masks = next(iter(dataloader))
    images = images.to(device)
    outputs = model(images)
    preds = (outputs > 0.5).float().cpu().numpy()
    images = images.cpu().numpy()
    masks = masks.cpu().numpy()

    for i in range(num_samples):
        image = np.transpose(images[i], (1, 2, 0))
        image = (image - image.min()) / (image.max() - image.min())  # Normalize for display
        mask = masks[i][0]
        pred = preds[i][0]

        fig, axs = plt.subplots(1, 3, figsize=(15, 5))
        axs[0].imshow(image)
        axs[0].set_title('Original Image')
        axs[0].axis('off')

        axs[1].imshow(mask, cmap='gray')
        axs[1].set_title('Ground Truth Mask')
        axs[1].axis('off')

        axs[2].imshow(pred, cmap='gray')
        axs[2].set_title('Predicted Mask')
        axs[2].axis('off')

        plt.show()

def plot_metrics(history, title_suffix=''):
    epochs = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(20, 5))

    # Loss plot
    plt.subplot(1, 4, 1)
    plt.plot(epochs, history['train_loss'], 'b', label='Training Loss')
    plt.plot(epochs, history['val_loss'], 'r', label='Validation Loss')
    plt.title('Loss over Epochs' + title_suffix)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    # Accuracy plot
    plt.subplot(1, 4, 2)
    plt.plot(epochs, history['train_acc'], 'b', label='Training Accuracy')
    plt.plot(epochs, history['val_acc'], 'r', label='Validation Accuracy')
    plt.title('Accuracy over Epochs' + title_suffix)
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # Dice Coefficient plot
    plt.subplot(1, 4, 3)
    plt.plot(epochs, history['train_dice'], 'b', label='Training Dice')
    plt.plot(epochs, history['val_dice'], 'r', label='Validation Dice')
    plt.title('Dice Coefficient over Epochs' + title_suffix)
    plt.xlabel('Epoch')
    plt.ylabel('Dice Coefficient')
    plt.legend()

    # IoU plot
    plt.subplot(1, 4, 4)
    plt.plot(epochs, history['train_iou'], 'b', label='Training IoU')
    plt.plot(epochs, history['val_iou'], 'r', label='Validation IoU')
    plt.title('IoU over Epochs' + title_suffix)
    plt.xlabel('Epoch')
    plt.ylabel('IoU')
    plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
def run_experiment(model_fn, optimizer_fn, scheduler_fn, augmentations, experiment_name, 
                   num_epochs=60, batch_size=8, patience=5):
    print(f"\nRunning experiment: {experiment_name}")
    
    # Load datasets with given augmentations
    train_dataset = KvasirInstrumentDataset(train_images_list, train_masks_list, augmentations=augmentations)
    valid_dataset = KvasirInstrumentDataset(test_images_list, test_masks_list, augmentations=no_augmentations)
    test_dataset  = KvasirInstrumentDataset(test_images_list, test_masks_list, augmentations=no_augmentations)

    # Data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    # Initialize model using the provided model function and move to device
    model = model_fn().to(device)
    
    # Create optimizer using provided function that accepts model parameters
    optimizer = optimizer_fn(model.parameters())
    
    # Initialize scheduler with the optimizer
    scheduler = scheduler_fn(optimizer)
    
    # Loss criterion (BCELoss in this example)
    criterion = nn.BCELoss()

    # Train the model with a custom training loop that supports early stopping, etc.
    history = train_model(
        model, 
        train_loader, 
        valid_loader,
        optimizer, 
        scheduler, 
        criterion,
        num_epochs=num_epochs, 
        patience=patience
    )
    
    # -------------------------------------------
    # After training, load the best model and evaluate on the test set
    best_model = model_fn().to(device)
    best_model.load_state_dict(torch.load(os.path.join(working_dir, 'best_model.pth')))
    best_model.eval()
    
    test_loss, test_dice, test_iou, test_acc = 0, 0, 0, 0
    with torch.no_grad():
        for images, masks in test_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = best_model(images)
            loss = criterion(outputs, masks)
            preds = (outputs > 0.5).float()
            dice, iou, acc = compute_metrics(preds, masks)
            test_loss += loss.item()
            test_dice += dice
            test_iou += iou
            test_acc += acc

    test_loss /= len(test_loader)
    test_dice /= len(test_loader)
    test_iou  /= len(test_loader)
    test_acc  /= len(test_loader)

    # Plot training metrics
    plot_metrics(history, title_suffix=f' ({experiment_name})')
    
    # Visualize predictions on the validation set
    visualize_predictions(model, valid_loader)
    
    # Determine the best epoch based on minimum validation loss
    best_epoch = min(range(len(history['val_loss'])), key=lambda i: history['val_loss'][i]) + 1

    # Collect key metrics from the best epoch
    best_metrics = {
        'Model': experiment_name,
        'Best Epoch': best_epoch,
        'Train Loss': history['train_loss'][best_epoch - 1],
        'Validation Loss': history['val_loss'][best_epoch - 1],
        'Train Accuracy': history['train_acc'][best_epoch - 1],
        'Validation Accuracy': history['val_acc'][best_epoch - 1],
        'Train Dice': history['train_dice'][best_epoch - 1],
        'Validation Dice': history['val_dice'][best_epoch - 1],
        'Train IoU': history['train_iou'][best_epoch - 1],
        'Validation IoU': history['val_iou'][best_epoch - 1],
        'Test Loss': test_loss,
        'Test Accuracy': test_acc,
        'Test Dice': test_dice,
        'Test IoU': test_iou
    }

    return history, best_metrics


In [ ]:
def get_unet():
    return UNet()

def get_unetpp():
    return UNetPlusPlus()

def get_fpn():
    return FPN()

In [ ]:

def get_unet():
    return UNet()

def get_unetpp():
    return UNetPlusPlus()

def get_fpn():
    return FPN()

optimizer_adam = lambda params: optim.Adam(params, lr=0.1)

scheduler_plateau = lambda optimizer: ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# Run experiments and store results
results = []

# Experiment 1: UNet with Adam, Plateau scheduler, and augmentations

# Experiment 3: FPN with RMSprop, Exponential scheduler, and augmentations
_, metrics3 = run_experiment(model_fn=get_fpn,
                             optimizer_fn=optimizer_adam,
                             scheduler_fn=scheduler_plateau ,
                             augmentations=no_augmentations,
                             experiment_name='FPN + Adam + Plateau + No Augmentation')
results.append(metrics3)

_, metrics1 = run_experiment(model_fn=get_unetpp,
                             optimizer_fn=optimizer_adam,
                             scheduler_fn=scheduler_plateau,
                             augmentations=no_augmentations,
                             experiment_name='UNet++ + Adam + Plateau + No Augmentation')
results.append(metrics1)

# Experiment 2: UNet++ with SGD, Step scheduler, no augmentation
_, metrics2 = run_experiment(model_fn=get_unet,
                             optimizer_fn=optimizer_adam,
                             scheduler_fn=scheduler_plateau,
                             augmentations=no_augmentations,
                             experiment_name='UNet + Adam + Plateau + No Augmentation')
results.append(metrics2)



# Create a summary table using pandas DataFrame
summary_df = pd.DataFrame(results)
print("\nSummary of Experiments:")
print(summary_df)

# Visualize the summary table using matplotlib
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=summary_df.values, colLabels=summary_df.columns, loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.auto_set_column_width(col=list(range(len(summary_df.columns))))
plt.title("Experiment Results Summary")
plt.show()

In [ ]:
optimizer_adam = lambda params: optim.Adam(params, lr=0.01)

scheduler_plateau = lambda optimizer: ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# Run experiments and store results
results = []

# Experiment 1: UNet with Adam, Plateau scheduler, and augmentations

# Experiment 3: FPN with RMSprop, Exponential scheduler, and augmentations
_, metrics3 = run_experiment(model_fn=get_fpn,
                             optimizer_fn=optimizer_adam,
                             scheduler_fn=scheduler_plateau ,
                             augmentations=augmentations,
                             experiment_name='FPN(0.01) +  Augmentation')
results.append(metrics3)

_, metrics1 = run_experiment(model_fn=get_unetpp,
                             optimizer_fn=optimizer_adam,
                             scheduler_fn=scheduler_plateau,
                             augmentations=augmentations,
                             experiment_name='UNet++(0.01) + Augmentation')
results.append(metrics1)

# Experiment 2: UNet++ with SGD, Step scheduler, no augmentation
_, metrics2 = run_experiment(model_fn=get_unet,
                             optimizer_fn=optimizer_adam,
                             scheduler_fn=scheduler_plateau,
                             augmentations=augmentations,
                             experiment_name='UNet(0.01) + Augmentation')
results.append(metrics2)



# Create a summary table using pandas DataFrame
summary_df = pd.DataFrame(results)
print("\nSummary of Experiments:")
print(summary_df)

# Visualize the summary table using matplotlib
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=summary_df.values, colLabels=summary_df.columns, loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.auto_set_column_width(col=list(range(len(summary_df.columns))))
plt.title("Experiment Results Summary")
plt.show()

In [ ]:
optimizer_adam = lambda params: optim.Adam(params, lr=1e-3)

scheduler_plateau = lambda optimizer: ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# Run experiments and store results
results = []

# Experiment 1: UNet with Adam, Plateau scheduler, and augmentations

# Experiment 3: FPN with RMSprop, Exponential scheduler, and augmentations
_, metrics3 = run_experiment(model_fn=get_fpn,
                             optimizer_fn=optimizer_adam,
                             scheduler_fn=scheduler_plateau ,
                             augmentations=augmentations,
                             experiment_name='FPN +  Augmentation')
results.append(metrics3)

_, metrics1 = run_experiment(model_fn=get_unetpp,
                             optimizer_fn=optimizer_adam,
                             scheduler_fn=scheduler_plateau,
                             augmentations=augmentations,
                             experiment_name='UNet++ + Augmentation')
results.append(metrics1)

# Experiment 2: UNet++ with SGD, Step scheduler, no augmentation
_, metrics2 = run_experiment(model_fn=get_unet,
                             optimizer_fn=optimizer_adam,
                             scheduler_fn=scheduler_plateau,
                             augmentations=augmentations,
                             experiment_name='UNet +  Augmentation')
results.append(metrics2)



# Create a summary table using pandas DataFrame
summary_df = pd.DataFrame(results)
print("\nSummary of Experiments:")
print(summary_df)

# Visualize the summary table using matplotlib
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=summary_df.values, colLabels=summary_df.columns, loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.auto_set_column_width(col=list(range(len(summary_df.columns))))
plt.title("Experiment Results Summary")
plt.show()